# TritFlow：OpenSUSI TR-1umで平衡三値HDLからGDSへ

三値のHDLから、製造用のレイアウトまで作れるでしょうか。三値論理には新しいデバイスやシミュレーションを使う研究が多く、オープンな設計ツールの[MRCS](https://github.com/aiunderstand/MixedRadixCircuitSynthesis)も、通常のCMOS向けには三値を二値へ変換します。TritFlowは、著者がテープアウトした三値セルをもとに、公開PDKのOpenSUSI TR-1um上で三値HDLをDRC/LVSの通ったGDSへ変換します。HDLを書き換えれば、自分の回路でも同じ工程を実行できます。

[OpenSUSI TR-1um](https://github.com/OpenSUSI/TR-1um)は日本の東海理化のプロセスを使う、NDA不要の1 µm CMOS PDKです。TritFlowは三値スタンダードセル、Verilog風の言語、合成、2層の配置配線、PDKのDRC/LVS、抽出後ngspiceをつなぎます。ライブラリとフローはApache-2.0で公開しています。

元のセルは2026年にテープアウト済みで、チップは2027年1月に届く予定です。実測はまだありません。採択された場合は、2027年2月14〜18日のISSCCポスターで、このセルの最初の実測結果をノートブックの予測と並べる計画です。

「**ランタイム → すべてのセルを実行**」を選ぶと、各セルが今の処理をログに出しながら進みます。最初に2トリット加算器のGDSができるので、拡大して素子と配線を見てください。8章では、自分で書き換えたHDLからGDSを作ります。





**図の4トリット加算器：** 全加算器4個をつないだHDL → 64セル → 1692 × 1590 µmのGDS。DRC 0件、LVS Match。

[![Colabで開く](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PenguinEino/tritflow/blob/colab-v0/notebook/tritflow_tr1um_ja.ipynb)

無料のColabでは、セットアップを含めて約5分で最後まで実行できます。セットアップの直後に止まったときは、もう一度「すべてのセルを実行」を選んでください。

<details><summary>著者とライセンス</summary>

- Name: Einosuke Okazaki (岡崎 英之介)
- Affiliation: Information and Communication Engineering Course, Department of Monozukuri
  Engineering, Tokyo Metropolitan College of Industrial Technology (KOSEN), Tokyo, Japan
- Status: Fifth-year student (final year of the five-year KOSEN engineering programme, which
  combines upper-secondary and the first two years of tertiary education)
- Email: einosukeokazaki@gmail.com
- GitHub: PenguinEino

Einosuke Okazaki will attend ISSCC 2027 in person and present the poster if selected (confirmed by the author, 2026-10-18).

Apache-2.0。著者が公開した元の回路図も含みます。

</details>


In [ ]:
#@title 準備：ソース、KLayout、ngspice、固定したTR-1um PDK { display-mode: "form" }
_TRITFLOW_SETUP_READY = False
globals().pop("_tf_cells", None)
import os, sys, json, time, hashlib, subprocess, importlib.util
from pathlib import Path
os.environ["TRITFLOW_NOTEBOOK_LANGUAGE"] = 'ja'
started = time.monotonic()
def step(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
SOURCE_URL = "https://github.com/PenguinEino/tritflow.git"
SOURCE_REF = os.environ.get("TRITFLOW_REF", "47bcac2086550444d9eef9fa65cc269a0c309931")
def source_git(*args):
    return subprocess.check_output(["git", "-C", str(ROOT), *args], text=True).strip()
if IN_COLAB:
    ROOT = Path("/content/tritflow")
    import re
    if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_REF):
        raise ValueError("TRITFLOW_REF must be an immutable full 40-character Git commit")
    if not ROOT.exists():
        step("GitHubからTritFlowのソースを取得")
        subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", "--filter=blob:none",
                        SOURCE_URL, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", SOURCE_REF], check=True)
        subprocess.run(["git", "-C", str(ROOT), "switch", "--detach", SOURCE_REF], check=True)
    if not (ROOT/".git").is_dir():
        raise RuntimeError("/content/tritflow must be a git clone; use a fresh Colab runtime")
    if source_git("remote", "get-url", "origin").removesuffix(".git") != SOURCE_URL.removesuffix(".git"):
        raise RuntimeError("existing checkout has a different origin; use a fresh Colab runtime")
    SOURCE_COMMIT = source_git("rev-parse", "--verify", "HEAD^{commit}")
    requested_commit = source_git("rev-parse", "--verify", SOURCE_REF+"^{commit}")
    if SOURCE_COMMIT != requested_commit or source_git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError("existing checkout differs from TRITFLOW_REF; use a fresh Colab runtime")
    print(f"Source: {SOURCE_URL} ref={SOURCE_REF} commit={SOURCE_COMMIT}", flush=True)
    if not (ROOT/"notebook/runtime_support.py").is_file():
        raise RuntimeError("TRITFLOW_REF predates this notebook's setup. Select its matching immutable release with TRITFLOW_REF in a fresh runtime.")
    sys.path.insert(0, str(ROOT/"notebook"))
    from runtime_support import SETUP_SCHEMA, setup_identity
    if SETUP_SCHEMA != "3":
        raise RuntimeError("The selected source commit predates the pinned runtime. Use the final submission commit with TRITFLOW_REF.")
    runtime = ROOT/"notebook/runtime.json"
    if (not runtime.is_file()
            or json.loads(runtime.read_text()).get("TRITFLOW_SETUP_SCHEMA") != SETUP_SCHEMA
            or json.loads(runtime.read_text()).get("TRITFLOW_SETUP_SHA256") != setup_identity(ROOT)):
        step("KLayout、ngspice、Pythonパッケージ、TR-1um PDKをインストール（初回のみ、約2分）")
        subprocess.run(["bash", str(ROOT/"notebook/colab_setup.sh"), "--source", str(ROOT)], check=True)
    else:
        step("ツールはこのランタイムにインストール済み")
    os.environ.update(json.loads(runtime.read_text()))
    sys.path.insert(0, os.environ["TRITFLOW_PYTHON_TARGET"])
else:
    candidates = [Path(os.environ.get("TRITFLOW_ROOT", ".")), Path.cwd(), Path.cwd().parent]
    ROOT = next((p.resolve() for p in candidates if (p/"pyproject.toml").is_file()
                 and (p/"tritflow").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("run locally from the full checkout or set TRITFLOW_ROOT")
    SOURCE_COMMIT = source_git("rev-parse", "--verify", "HEAD^{commit}") if (ROOT/".git").exists() else "tracked-tree export (no .git)"
    print(f"Source: {ROOT} commit={SOURCE_COMMIT}")
runtime = ROOT/"notebook/runtime.json"
if not IN_COLAB and runtime.is_file():
    os.environ.update(json.loads(runtime.read_text()))
    sys.path.insert(0, os.environ["TRITFLOW_PYTHON_TARGET"])
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/"notebook"))
sys.path.append(str(ROOT/"figures"))   # the paper's figure scripts, reused by method cells
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
from runtime_support import check_runtime, print_tool_versions
step("Pythonパッケージ、KLayout、ngspiceを確認")
check_runtime()
print_tool_versions()
_TRITFLOW_SETUP_READY = True
step(f"準備完了：{time.monotonic()-started:.0f} s")


In [ ]:
#@title 準備：モジュールの読込みと実行状態 { display-mode: "form" }
NARRATIVE_CONTEXT_SHA256 = '54a8cbe63baefc8fd74fa9cb6fca6b8be0938ab6ccc874dc40dd2922bbe1f376'
NOTEBOOK_LANGUAGE = 'ja'

globals().pop("_tf_cells", None)
if not globals().get("_TRITFLOW_SETUP_READY", False):
    raise RuntimeError("Run the first setup cell successfully, then rerun the imports cell. If setup requested a restart, restart once first.")
from runtime_support import CellState, step
step("TritFlowを読み込み、固定した入力データを確認")
import numpy as np
from IPython.display import display, HTML, Image
from tritflow import flow, CELLS_DIR, PDK_ROOT, PASS_BAND, TRIT_VOLTS
from tritflow.sim import dc_transfer, input_vectors, subcircuits
from tritflow.viz.graph import truth_table_html
from tritflow.viz.waves import waveform_figure, waveform_scope
from tritflow.viz.layout import render_gds, placement_animation, routing_progress
from tritflow.viz.cell import device_mapping
from tritflow.sim.provenance import pdk_provenance
from tritflow.sim.postlayout import logical_nodeset
from cache import Cache, runtime_fingerprint, source_manifest
from campaign import RecordedCampaign
from schematics import schematic_gallery
from experiments import (complete_design, experiment_artifacts, assert_experiment,
                         load_edited, verify_publication_sources, adder_demo_vectors)
EXTENDED = os.environ.get("TRITFLOW_EXTENDED", "0") == "1"
BUILD = Path(os.environ.get("TRITFLOW_NOTEBOOK_BUILD", ROOT/"build/notebook")).resolve()
qualified_pdk = pdk_provenance(PDK_ROOT)
assert qualified_pdk["qualified_commit"] == json.loads((ROOT/"notebook/setup_pins.json").read_text())["pdk_commit"]
native_sources = source_manifest(ROOT,PDK_ROOT)
cache = Cache(BUILD/"cache", runtime_fingerprint(ROOT, PDK_ROOT,names=native_sources),
              force=os.environ.get("TRITFLOW_FORCE_RUN", "0") == "1",sources=native_sources)
recorded_campaign = RecordedCampaign(ROOT/"notebook/assets/four_designs.zip",ROOT,PDK_ROOT,
                                     enabled=not cache.force)
renderer_sha256 = hashlib.sha256((ROOT/"tritflow/viz/layout.py").read_bytes()).hexdigest()
from presentation import show, table, evidence, gds_download
from presentation import configure_language, label as display_label, paper_figure
configure_language(globals().get("NOTEBOOK_LANGUAGE", "en"))
from presentation import localized_display as display

from IPython.display import Markdown, Code
from editorial import context, context_identity, render as render_narrative, dimensions
from paper_evidence import load_paper
from learning import sizing_slider, value_selector, highlight_devices, cell_cards
from predictions import divider, zero_dc, rail_power
from presentation import details_html, details_svg
from routing_identity import routing_label, archive_routing
paper = load_paper(ROOT)
narrative_values = context(ROOT)
assert context_identity(narrative_values) == NARRATIVE_CONTEXT_SHA256, "Published figure measurements changed; regenerate the notebook before running."
step("準備完了")

_tf_cells = CellState({'setup': 'setup', 'imports': 'imports', 'opening_inputs': '0章：試験条件と独立した整数の参照関数', 'method-opening': 'method-opening', 'opening': '0章：論理、DRC/LVS、抽出後SPICEの照合', 'method-gds': 'method-gds', 'method-gates': 'method-gates', 'gates': '1章：真理値とDC特性の照合、元のINV回路図', 'method-completeness': 'method-completeness', 'completeness': '1章：1入力関数の閉包とSłupeckiの判定法', 'method-sizing': 'method-sizing', 'sizing': '2章：MOS寸法を固定した抵抗長の比較', 'method-cells': 'method-cells', 'cells': '3章：共通寸法、クランプ素子、セルのDRC/LVS', 'method-arithmetic': 'method-arithmetic', 'arithmetic': '4章：全入力の加算と元の回路図のセル数', 'method-synthesis': 'method-synthesis', 'synthesis': '5章：割当規則とゲートレベルTritVerilog', 'method-schematic': 'method-schematic', 'method-pnr': 'method-pnr', 'pnr': '6章：配置配線の詳細と任意のフロアプラン実験', 'method-pre': 'method-pre', 'pre': '7章：同じ素子で配線容量の有無を比較', 'published': '7章：保存済みの桁数・負荷掃引', 'method-sta': 'method-sta', 'sta': '7章：STAの比較表とクリティカルパス', 'method-edit': 'method-edit', 'method-pick': 'method-pick', 'method-custom': 'method-custom', 'custom_check': '8章：全入力を整数の参照関数と照合', 'method-custom_layout': 'method-custom_layout', 'custom': '8章：編集したGDSの抽出後SPICEと波形', 'method-broken': 'method-broken', 'method-shuttle': 'method-shuttle', 'shuttle': '9章：保存済みのトップ検査とパッド図', 'characterization': '付録B：セル特性とファンアウトの試験', 'four': '付録B：4回路の物理検査と遅延の比較', 'final': '付録B：検証表、入力データ、実行時間'})


## 0. HDLから配線済みGDSへ

図は4トリット加算器の全工程です。下のコードでは2トリット版を作り、合成、配置配線、DRC/LVS、抽出後SPICEを続けて実行します。その直後のビューアで配線済みGDSを開き、同じファイルをダウンロードできます。

トリットの値−1、0、+1を、それぞれ-5、0、+5 Vで表します。HDLの`trit`は、この3値をとる信号の型です。

下の2トリット加算器は、全加算器を2個つないだだけの回路です。下の桁の桁上がり`c1`を上の桁へ渡します。全加算器の中身は4章で見ます。


`examples/adder2.tv`

```verilog
`include "full_add.tv"
// Two-trit adder: digit i has weight 3**i, so A + B + cin = SUM + 9*cout.
// The carry c1 passes from the lower full adder to the upper one.
module adder2(input trit [1:0] a, b, input trit cin,
              output trit [1:0] sum, output trit cout);
  wire trit c1;
  full_add fa0 (.a(a[0]), .b(b[0]), .cin(cin), .sum(sum[0]), .cout(c1));
  full_add fa1 (.a(a[1]), .b(b[1]), .cin(c1),  .sum(sum[1]), .cout(cout));
endmodule
```


In [ ]:
#@title 0章：試験条件と独立した整数の参照関数 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('opening_inputs', '0章：試験条件と独立した整数の参照関数', requires=[]):
    LOAD_F, HOLD_S, STEP_S, EDGE_S, TEMP_C = 100e-15, 1e-6, 2e-9, 1e-9, 27
    PNR_ROWS, PNR_ANNEAL_STEPS = None, 600  # automatic near-square floorplan; placement effort
    CONDITIONS = dict(hold_s=HOLD_S, step_s=STEP_S, edge_s=EDGE_S, temperature_c=TEMP_C, timeout=600)
    out = BUILD/"adder2_demo"
    def adder_reference(vector):
        """Independent balanced-ternary integer oracle; `inputs` comes from the method cell."""
        v = dict(zip(inputs, vector, strict=True))
        remainder = sum((v[f"a[{i}]"] + v[f"b[{i}]"])*3**i for i in range(2)) + v["cin"]
        result = {}
        for i in range(2):
            result[f"sum[{i}]"] = (remainder+1)%3-1
            remainder = (remainder-result[f"sum[{i}]"])//3
        return {**result, "cout": remainder}


In [ ]:
step("1/5 TritVerilogの加算器を読み込み")
d = flow.load(ROOT/"examples/adder2.tv", "adder2")                # 1. TritVerilog source
step("2/5 合成：式をセルへ割り当て、全入力で一致を確認")
net = d.synthesize(name="adder2_demo")                           # 2. cells + exhaustive equivalence check
inputs = [p.name for p in net.ports if p.dir == "input"]
vectors, vector_labels = adder_demo_vectors(inputs)               #    three-level test sequence
step("3/5 2層メタルで配置配線")
lay = net.place_and_route(output_dir=out, seed=0, rows=PNR_ROWS, anneal_steps=PNR_ANNEAL_STEPS)  # 3. GDS
step("4/5 TR-1umのデッキでDRCとLVS")
drc, lvs = lay.drc(), lay.lvs()                                  # 4. TR-1um DRC deck, strict LVS
guesses = logical_nodeset(net, lvs["report"], inputs, vectors[0])
step("5/5 レイアウトを抽出し、ngspiceでシミュレーション（約30秒）")
post = lay.extract().simulate(vectors, load=LOAD_F, expected=adder_reference,
                              nodeset=guesses, **CONDITIONS)     # 5. extracted devices + wire C in ngspice
print(f"DRC合格： {drc['clean']} · LVS一致： {lvs['matched']} · アナログ合格： {post['measurements']['passed']}")


In [ ]:
#@title 0章：論理、DRC/LVS、抽出後SPICEの照合 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('opening', '0章：論理、DRC/LVS、抽出後SPICEの照合', requires=[]):
    logical = d.truth_table()
    outputs = [p.name for p in net.ports if p.dir == "output"]
    assert all(len(v) == len(inputs) for v in vectors)
    assert len(logical[inputs[0]]) == 3**len(inputs)
    expected_sequence = [adder_reference(v) for v in vectors]
    assert all({r[p] for r in expected_sequence} == {-1,0,1} for p in outputs)
    assert all(a != b for a,b in zip(vectors,vectors[1:]))
    for row in zip(*(logical[name] for name in inputs)):
        assert adder_reference(row) == {k: int(v) for k,v in d.design.evaluate(dict(zip(inputs,row))).items()}
    assert drc["clean"] and lvs["matched"], (drc, lvs)
    assert post["measurements"]["passed"], post["measurements"]
    demo = {**post, "stats": lay.stats, "drc": drc, "lvs": lvs, "initial_nodeset_v": guesses}
    assert not lay.stats["library_mock"]
    assert all({row["outputs"][p]["ideal_trit"] for row in demo["measurements"]["rows"]} == {-1,0,1}
               and demo["measurements"]["max_settling_s"][p] > 0 for p in outputs)
    table([["実行中の加算器", "セル数", "寸法（µm）", "DRC件数", "厳密なLVS", "アナログ"],
           [net.top,net.report()["total_cells"],dimensions(lay.stats["die_size_um"]),
            demo["drc"]["violations"],demo["lvs"]["matched"],demo["measurements"]["passed"]]])
    print(f"整数の足し算との照合：全{len(logical[inputs[0]])}入力状態が一致。SPICEではそのうち{len(set(vectors))}状態を順に通します。")


In [ ]:
from tritflow.viz.layout import gds_viewer
step("配線済みGDSをビューアで表示")
display(gds_viewer(lay.gds_path, top=net.top))             # zoom to the actual routed cells
gds_download(lay.gds_path)                               # download the checked GDSII


## 1. CMOSで作る三値ゲート

INVの中間出力はVDD側とVSS側の抵抗分圧で決まります。入力が0のときは両側のMOSが導通し、入力を固定しても電流が流れます。NANYにはVMIDへのクランプ経路があり、符号が逆の入力に対して出力を0 Vに保ちます。下のセルでは真理値表とテープアウトした回路図を切り替え、続いてDC特性を計算します。

$$
\begin{aligned}
\mathrm{INV}(a)&=-a, & \mathrm{NAND}(a,b)&=-\min(a,b),\\
\mathrm{NOR}(a,b)&=-\max(a,b), & \mathrm{NANY}(a,b)&=-\mathrm{clamp}(a+b,-1,1).
\end{aligned}
$$

### 分圧の予測

INVの入力を0 Vに置き、導通中のMOSをオン抵抗で近似します。直列抵抗を含む上側と下側の抵抗を$R_u$、$R_d$とすると、

$$
I_{\mathrm{hand}}=\frac{V_{DD}-V_{SS}}{R_u+R_d},\qquad
V_{Y,\mathrm{hand}}=V_{SS}+I_{\mathrm{hand}}R_d,\qquad
P_{\mathrm{hand}}=(V_{DD}-V_{SS})I_{\mathrm{hand}}.
$$

両側の抵抗が等しければ出力は0 Vです。寸法とPDKモデルから求めた$R_u$=11.33 kΩ、$R_d$=11.23 kΩを使うと、出力-0.02066 V、電流0.4432 mA、電力4.432 mWと予測します。次に非線形の素子モデルを使うSPICEと比べます。



`examples/gates.tv`

```verilog
// The four primitive cells, written as TritVerilog expressions.
// Trits are -1, 0, +1; '~' negates, '&' is min, '|' is max.
module gates(input  trit a, b,
             output trit inva, nandab, norab, nanyab);
  assign inva   = ~a;          // INV  : -a
  assign nandab = ~(a & b);    // NAND : -min(a, b)
  assign norab  = ~(a | b);    // NOR  : -max(a, b)
  assign nanyab = nany(a, b);  // NANY : -sat(a + b)
endmodule
```


In [ ]:
from fig_gates import simulate, draw                      # the paper's own Fig. 2 script
from paperstyle import save as save_figure
step("4種類のゲートの真理値表")
gates = flow.load(ROOT/"examples/gates.tv", "gates")       # the definition shown above
t = gates.truth_table()                                    # all nine input pairs, ideal trits
display(cell_cards(t, CELLS_DIR/"out", ROOT/"notebook/assets/schematics"))  # truth tables + taped-out schematics
step("抽出したセルのDC掃引10本をngspiceで計算")
curves = simulate(CELLS_DIR/"out")                         # ten ngspice DC sweeps of the extracted layouts
png = save_figure(draw(curves), "fig_gates", out=BUILD/"figures")
display(paper_figure(png, width=1000))                      # Fig. 2, regenerated from your own run
v_a, v_y, i_ma = curves["INV"][0]
i0 = int(np.argmin(abs(v_a)))
print(f"A = 0 V:  Y = {1e3*v_y[i0]:.1f} mV,  電源電流 = {i_ma[i0]:.3f} mA")


抽出後セルのDC伝達特性とVDD電流。横軸は入力A、2入力セルのBは各論理レベルに固定します。帯は出力の許容範囲±0.5 Vです。

> **ここから分かること。** 抽出後のINVは入力0 V、27 °Cで、出力-0.03954 V、VDD電流0.4072 mA、全電源の合計電力4.072 mWです。分圧の手計算に対応する回路図SPICEでは、出力-0.03954 V、電力4.072 mWとなります。

### NANYと定数セル

二値のNANDと違い、三値のINV・NAND・NORは定数3つを加えても、1入力関数の一部しか作れません。下のコードは入力線と定数から始めてゲートを合成し、新しい真理値表が増えなくなるまで繰り返します。生成できる1入力関数はINV・NAND・NORと定数3つで11/27、NANYとTIEZだけで3/27、NANYとTIEP/TIENで27/27です。

[Słupeckiの判定法](https://www.conferences.info.uaic.ro/mathematical-foundations-of-informatics/2019/PDF/MFOI2019_Volume.pdf)では全1入力関数を作れ、かつ両入力に本質的に依存して3値すべてを出す2入力関数があれば機能的完全です。NANYは後者を満たすので、NANYとTIEP/TIENの組は機能的完全です。NANYは$f(-a,-b)=-f(a,b)$を満たす奇関数で、0だけを足して合成しても奇関数のままです。±1の定数がこの制約を外します。

算術にも向いていて、半加算器はNANY 5個とINV 2個でできます。この数も、下のセルでHDLを合成して数えます。なお、完全なのはセルの組です。TritFlowの合成は書いた式をセルへ割り当てるもので、真理値表から回路を探すわけではありません。


In [ ]:
#@title 1章：真理値とDC特性の照合、元のINV回路図 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('gates', '1章：真理値とDC特性の照合、元のINV回路図', requires=[]):
    assert np.array_equal(t["inva"], -t["a"])
    assert np.array_equal(t["nandab"], -np.minimum(t["a"], t["b"]))
    assert np.array_equal(t["norab"], -np.maximum(t["a"], t["b"]))
    assert np.array_equal(t["nanyab"], -np.clip(t["a"]+t["b"],-1,1))
    for name in ["INV","NAND","NOR","NANY"]:
        body = subcircuits((CELLS_DIR/f"out/{name}/{name}.spice").read_text())[name.lower()]["body"]
        assert ("VMID" in body.upper()) == (name == "NANY"), name
    # The regenerated figure uses the same public DC call and zero-input sample as the paper.
    assert {cell: sorted(c) for cell, c in curves.items()} == {"INV": [0], "NAND": [-1, 0, 1], "NOR": [-1, 0, 1], "NANY": [-1, 0, 1]}
    assert abs(v_a[i0]) < 1e-9
    assert abs(v_y[i0]-paper["inv_dc"]["output_v"]) < 1e-8
    # The held input really is held: at A = +5 V, NAND gives +5 V for B = −1 and −5 V for B = +1.
    assert curves["NAND"][-1][1][-1] > 4.5 and curves["NAND"][1][1][-1] < -4.5
    details_html("テープアウトしたINVの回路図とソース",schematic_gallery(
        ROOT/"notebook/assets/schematics",names=["INV"],selected="INV"))


In [ ]:
from completeness import unary_closure, inv, nand, nor, nany
bases = [("INV/NAND/NOR + TIEP/TIEZ/TIEN", (inv,), (nand, nor), (-1, 0, 1), 11),
         ("NANY + TIEZ", (), (nany,), (0,), 3),
         ("NANY + TIEP/TIEN", (), (nany,), (-1, 1), 27)]
step("新しい1入力関数が出なくなるまでゲートを組み合わせる")
closures = {}
for basis_name, unary, binary, constants, expected in bases:
    closures[basis_name] = unary_closure(unary=unary, binary=binary, constants=constants)
    assert len(closures[basis_name]) == expected
    print(f"{basis_name}: {len(closures[basis_name])}/27 通りの1入力関数")
step("合成した半加算器のセルを数える")
half_counts = flow.load(ROOT/"examples/half_add.tv", "half_add").synthesize().report()["cells"]
print("half_add:", half_counts)


In [ ]:
#@title 1章：1入力関数の閉包とSłupeckiの判定法 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('completeness', '1章：1入力関数の閉包とSłupeckiの判定法', requires=[]):
    from itertools import product
    from completeness import TRITS, surjective_and_depends_on_both
    assert set(closures[bases[2][0]]) == set(product(TRITS, repeat=3))
    assert surjective_and_depends_on_both(nany)
    assert all(nany(-a, -b) == -nany(a, b) for a, b in product(TRITS, repeat=2))
    assert all(f[1] == 0 and f[0] == -f[2] for f in closures[bases[1][0]])
    assert half_counts == {"NANY": 5, "INV": 2}
    table([["セルの組", "1入力関数の数", "全1入力関数"]] +
          [[label, len(closures[label]), len(closures[label]) == 27] for label, *_ in bases])


## 2. 中間レベルと静的電力を調整する

プルダウンを強くすれば中間出力は下がり、プルアップを強くすれば上がります。両側の直列抵抗を長くすると電流は減りますが、出力誤差も変わります。図の丸印はテープアウトした設計の採用寸法です。スライダーでW/Lと抵抗長を選び、伝達特性と誤差・電流の点を動かしてください。

### 寸法からの予測

導通中のMOSのオン抵抗を$W/L$に反比例すると近似し、前章の分圧式に戻して計算します。

$$
r_{\mathrm{on}}'\simeq r_{\mathrm{on}}\frac{(W/L)_{\mathrm{adopted}}}{(W/L)'},\qquad
\Delta V_Y\simeq(V_{DD}-V_{SS})\frac{R_u\Delta R_d-R_d\Delta R_u}{(R_u+R_d)^2}.
$$





INVの抵抗長20, 30, 45 µmでの伝達特性と、INV/NANYの最大レベル誤差・電流。丸印が採用寸法、塗りつぶしと白抜きが許容範囲の内外です。逆三角は表示下限3 mV未満を示します。

> **ここから分かること。** 採用したINVの最大レベル誤差は39.54 mV、最大電源電流の絶対値は407.2 µAです。NANYではそれぞれ44.6 mV、421.4 µAとなります。加算器では1セル当たり約4.169 mWを使い、6トリットでは400.2 mWになります。

MOS寸法を固定し、INVの両抵抗を30から45 µmに伸ばすと、最大電流は407.2から260.7 µAへ減り、最大誤差は39.54から52.44 mVへ増えます。電流を減らせば中間レベルの精度が落ちる、という綱引きです。

この電流は入力が変わらなくても流れ続けます。二値CMOSと違い、動作周波数を下げても、中間レベルを保つ電力はなくなりません。三値CMOSに特有のコストです。


In [ ]:
from tritflow.sim.sizing import load_grid
step("計算済みのW/L・抵抗長の掃引を読み込み")
sizing_grids = {cell: load_grid(cell) for cell in ("INV", "NANY")}  # precomputed W/L/R sweeps
gi = sizing_grids["INV"]
adopted = gi["rows"][gi["adopted_index"]]                           # the taped-out size
hand = divider(adopted["dimensions"], PDK_ROOT)                     # predict first ...
table([["入力0のINV", "手計算", "SPICE"],
       ["出力（V）", hand["output_v"], zero_dc(adopted)["output_v"]["Y"]],
       ["静的電力（mW）", hand["power_w"]*1e3, rail_power(zero_dc(adopted), gi["conditions"]["supplies_v"])*1e3]])
show(sizing_slider(gi, PDK_ROOT))                                   # ... then explore every simulated size


In [ ]:
#@title 2章：MOS寸法を固定した抵抗長の比較 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('sizing', '2章：MOS寸法を固定した抵抗長の比較', requires=[]):
    # Hold MOS sizes fixed: these are actual simulated RR candidates.
    rr_candidates = sorted((r for r in gi["rows"] if r["dc"] and
        all(r["dimensions"][k] == adopted["dimensions"][k] for k in ("MP","MN"))),
        key=lambda r:r["dimensions"]["RP"]["l_um"])
    table([["共通の抵抗長（µm）","最大レベル誤差（mV）","最大電源電流の絶対値（µA）","参照"]]+[
        [r["dimensions"]["RP"]["l_um"],r["max_error_v"]*1e3,r["max_static_current_a"]*1e6,
         "taped out" if r["is_adopted"] else ""] for r in rr_candidates])


## 3. テープアウトした回路を並べられるセルにする

配置配線には、回路に加えてセル枠、ピンの接続位置、配線の障害物が必要です。TritFlowのセルは高さをそろえ、VDD・VMID・VSSを横に通します。図では元のNANYとスタンダードセルを同じ縮尺で比べ、隣接セルへ続く3本のレールを確認できます。

内部のM2とビアも障害物に含めます。信号ピンはM1上に置き、縦のM2からビアで接続します。許す向きは通常配置と左右反転で、レール順は変わりません。TIEは抵抗を介して定数を供給し、FILLは隙間のレールとウェルをつなぎます。





元のNANY、共通高さに作り直したNANY、突き当てた4セルの行。素子のグループと3本の電源レールを示します。

> **ここから分かること。** 元のNANYは148 × 115 µm、スタンダードセルは168 × 132 µmで、面積比は1.303倍です。回路と素子寸法を保ちながら、共通レールとピン接続位置を揃えました。図の4セルを通して電源レールが続きます。

下の図ではクランプMOSとGDS上の素子を対応づけます。セル単体と突き当て配置のDRC/LVSは付録Bにまとめています。


In [ ]:
step("スタンダードセルのライブラリを読み込み")
library = json.loads((CELLS_DIR/"out/library.json").read_text())
abstracts = {c["name"]: json.loads((CELLS_DIR/"out"/c["abstract"]).read_text()) for c in library}
print({name: f"{a['width']:g} × {a['height']:g} µm" for name, a in abstracts.items()})
step("NANYの回路図の素子をLVSでGDSと対応づけ")
mapping = device_mapping("NANY", selected="XCN1")       # LVS-matched schematic devices on the real GDS
show(highlight_devices(mapping, {"XCP1", "XCN1"},
                       title="NANY: Y → XCP1/XM6 → c1 → XCN1/XM7 → VMID, A=−1 B=+1"))


In [ ]:
#@title 3章：共通寸法、クランプ素子、セルのDRC/LVS { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('cells', '3章：共通寸法、クランプ素子、セルのDRC/LVS', requires=[]):
    template = json.loads((CELLS_DIR/"template.json").read_text())
    generation = json.loads((CELLS_DIR/"out/generation.json").read_text())
    assert {a["height"] for a in abstracts.values()} == {template["height"]}
    assert generation["height_um"] == template["height"]

    cell_dimension_rows = ([["Cell", "Width × common height (µm)", "Area (µm²)", "Free full-height M2 tracks"]]+
          [[name, [a["width"],a["height"]], a["width"]*a["height"], a["free_M2_tracks"]["count"]]
           for name,a in abstracts.items()])
    abstract = json.loads((CELLS_DIR/"out/NANY/NANY.abstract.json").read_text())
    # Highlight both conducting devices of the zero-level clamp. These are device
    # outlines, not an invented extracted current-density or wire-current map.
    clamp_devices = {"XCP1","XCN1"}
    clamp_body = subcircuits((CELLS_DIR/"out/NANY/NANY.spice").read_text())["nany"]["body"]
    clamp_terminals = {fields[0]:fields[1:5] for fields in map(str.split,clamp_body.splitlines()) if fields}
    assert clamp_terminals["XCP1"] == ["c1","A","Y","VDD"]
    assert clamp_terminals["XCN1"] == ["c1","B","VMID","VSS"]
    clamp_character = json.loads((CELLS_DIR/"out/NANY/NANY.char.json").read_text())
    assert hashlib.sha256((CELLS_DIR/"out/NANY/NANY.ext.spice").read_bytes()).hexdigest() == clamp_character["source_sha256"]
    clamp_dc = next(r for r in clamp_character["dc"] if r["input_trits"] == [-1,1])
    assert clamp_dc["passed"] and abs(clamp_dc["output_v"]["Y"]) <= PASS_BAND
    assert clamp_devices <= {r["name"] for r in mapping.rows}
    evidence("Device correspondence and W/L",mapping.rows)

    qualification = CELLS_DIR/"out/verify_report.json"
    if qualification.is_file():
        report = json.loads(qualification.read_text())
        assert report["passed"]
        assert all(hashlib.sha256((PDK_ROOT/name).read_bytes()).hexdigest() == value
                   for name,value in report["deck_sha256"].items()), "cell qualification uses a different PDK"
        for cell, artifacts in report["artifact_sha256"].items():
            for suffix, expected_hash in artifacts.items():
                assert hashlib.sha256((CELLS_DIR/f"out/{cell}/{cell}.{suffix}").read_bytes()).hexdigest() == expected_hash
        cell_qualification_rows = ([["Cell", "Recorded DRC clean", "Recorded LVS qualification", "LVS method"]]+[
            [name, result["drc"]["passed"], result["lvs"]["passed"],
             result["lvs"].get("method","strict device/port comparison")]
            for name,result in report["cells"].items()])
        if "FILL" in report["cells"]:
            evidence("FILL qualification", report["cells"]["FILL"]["lvs"]["explanation"])
        evidence("Recorded qualification scope", report["scope"])
    else:
        raise FileNotFoundError("Required pinned-PDK cell qualification: "+str(qualification))


## 4. TritVerilogで符号付き加算を書く

平衡三値の第$i$桁の重みは$3^i$です。各桁に−1を使えるので、別の符号ビットなしで正負の整数を表せます。半加算器は2入力、全加算器は桁上がりも加え、次の関係を満たします。

$$
a+b+c_{in}=s+3c_{out},\qquad
A+B+c_{in}=\sum_{i=0}^{N-1}s_i3^i+c_{out}3^N.
$$

例えば$1+1+0=(-1)+3\cdot(+1)$です。下のHDLでは、全加算器が半加算器を2回使います。0章の2トリット加算器は、この全加算器を2個つないだものです。真理値表は、ゲートの式とは別に計算した整数の足し算と照合します。

書き方はVerilogとほぼ同じです。`~`は符号反転、`&`はmin、`|`はmax、`nany`は飽和加算の符号反転を表します。





半加算器と全加算器のHDL、半加算器のセル、全加算器の階層。配線色は入力に対する定常トリット値で、青が−1、灰色が0、赤が+1です。

> **ここから分かること。** 合成した半加算器はNANY 5個とINV 2個、全加算器はNANY 11個とINV 5個です。全加算器は半加算器の桁上がりをNANYとINVでまとめます。


`examples/half_add.tv`

```verilog
// Reference: balanced-ternary-logic/HALF_ADDER_SIMULATION.md.
// a + b = sum + 3*carry. Exactly five NANY and two INV cells.
module half_add(input trit a, b, output trit sum, carry);
  wire trit t, u, na, d, nd;
  nany g_t (t, a, b);
  nany g_u (u, b, t);
  inv g_na (na, a);
  nany g_c (carry, na, u);
  nany g_d (d, t, carry);
  inv g_nd (nd, d);
  nany g_s (sum, nd, carry);
endmodule
```

`examples/full_add.tv`

```verilog
`include "half_add.tv"
// Reference: balanced-ternary-logic/FULL_ADDER.md.
// a + b + cin = sum + 3*cout. Exactly eleven NANY and five INV.
module full_add(input trit a, b, cin, output trit sum, cout);
  wire trit s1, c1, c2, nc;
  half_add ha1 (.a(a), .b(b), .sum(s1), .carry(c1));
  half_add ha2 (.a(s1), .b(cin), .sum(sum), .carry(c2));
  nany g_nc (nc, c1, c2);
  inv g_cout (cout, nc);
endmodule
```


In [ ]:
ha = flow.load(ROOT/"examples/half_add.tv", "half_add")
fa = flow.load(ROOT/"examples/full_add.tv", "full_add")
step("全加算器を27通りの全入力で確認")
fa_table = fa.truth_table()                                     # all 27 input states
assert np.array_equal(fa_table["a"] + fa_table["b"] + fa_table["cin"], fa_table["sum"] + 3*fa_table["cout"])
step("半加算器と全加算器を合成")
ha_counts, fa_counts = ha.synthesize().report()["cells"], fa.synthesize().report()["cells"]
print("half_add:", ha_counts, "  full_add:", fa_counts)


In [ ]:
#@title 4章：全入力の加算と元の回路図のセル数 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('arithmetic', '4章：全入力の加算と元の回路図のセル数', requires=[]):
    details_html("半加算器の全入力真理値表",truth_table_html(ha.design))
    details_html("全加算器の全入力真理値表",truth_table_html(fa.design))
    ha_table = ha.truth_table()
    assert np.array_equal(ha_table["a"]+ha_table["b"], ha_table["sum"]+3*ha_table["carry"])
    schematic_sources = json.loads((ROOT/"notebook/assets/schematics/manifest.json").read_text())["schematics"]
    ha_symbols = schematic_sources["half_adder"]["instances_by_symbol"]
    fa_symbols = schematic_sources["full_adder"]["instances_by_symbol"]
    symbol_names = {"INV":"inverter", "NANY":"nany"}
    assert ha_counts == {cell:ha_symbols[symbol] for cell,symbol in symbol_names.items()}
    assert fa_counts == {cell:fa_symbols["half_adder"]*ha_counts[cell]+fa_symbols[symbol]
                         for cell,symbol in symbol_names.items()}
    table([["回路", "セル数", "全入力状態数"],
           ["half_add",ha_counts,len(ha_table["sum"])], ["full_add",fa_counts,len(fa_table["sum"])]])
    evidence("2トリット加算器のHDL", (ROOT/"examples/adder2.tv").read_text())


## 5. 式からセルへの変換を追う

合成は階層を展開し、三値で成り立つ規則で式を整理してセルへ割り当てます。下の図は合成した半加算器です。入力の値を選ぶと配線の色が変わり、青が−1、灰色が0、赤が+1を表します。

`a = b = +1`を選び、途中の信号を出力まで追ってください。和2は、下位桁−1と桁上がり+1になります。

否定したminはNANDへ、minはNANDとINVへ写します。maxにはNORを使い、定数はTIEへ写します。二重反転、共通式、未使用回路を整理し、必要なバッファを挿入した結果を元のHDLと照合します。三値では$\min(a,-a)$も$\max(a,-a)$も$a=0$で0なので、二値の相補則は使えません。

> **ここから分かること。** 半加算器の7セルは、入力を変えても整数の和を保ちます。この同じネットリストを配置配線の入力とし、LVSの回路図側の比較対象にも使います。


In [ ]:
step("全加算器の合成：式の整理、セルへの割当、全入力の照合")
fa_net = fa.synthesize()                         # rewrite → map to cells → equivalence check
print(fa_net.report()["equivalence"])
half_net = flow.load(ROOT/"examples/half_add.tv", "half_add").synthesize()
step("合成した半加算器を描画")
display(value_selector(half_net, title=display_label("Mapped half adder: choose the input trits")))  # settled nets


In [ ]:
#@title 5章：割当規則とゲートレベルTritVerilog { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('synthesis', '5章：割当規則とゲートレベルTritVerilog', requires=[]):
    assert fa_net.report()["equivalence"]["passed"]
    assert fa_net.report()["equivalence"]["vectors"] == len(fa_table["sum"])
    table([["Report field", "Value"]]+[[k,v] for k,v in fa_net.report().items()],details="Full synthesis report")
    from tritflow.synth import tritverilog
    evidence("Gate-level TritVerilog", tritverilog(fa_net))
    mapping_cases = [("~a",["INV"]), ("~(a&b)",["NAND"]), ("a&b",["NAND","INV"]),
                     ("~(a|b)",["NOR"]), ("a|b",["NOR","INV"]), ("nany(a,b)",["NANY"]),
                     ("1'tT",["TIEN"]), ("1't0",["TIEZ"]), ("1't1",["TIEP"])]
    mapping_rows = [["HDL expression", "Computed cells"]]
    for expr, expected_cells in mapping_cases:
        mapped = flow.loads(f"module mapping(input trit a,b,output trit y); assign y={expr}; endmodule").synthesize()
        assert sorted(inst.cell for inst in mapped.instances) == sorted(expected_cells)
        mapping_rows.append([expr, mapped.report()["cells"]])
    table(mapping_rows,details="Expression-to-cell mapping checks")


In [ ]:
step("合成した2トリット加算器を回路図に描画")
sch = net.schematic(inputs={"a[0]": 1, "a[1]": 0, "b[0]": 1, "b[1]": 0, "cin": 0}, net_names=True,
                    title="2トリット加算器：合成した階層と定常値")
display(sch)                                     # the synthesized netlist, drawn as a schematic


## 6. 2層の配線でGDSを完成させる

セル行の間に配線チャネルを設け、焼きなまし法で配線長を縮めてから格子上で接続します。M1の信号配線はチャネル内を横に通り、M2は障害物のない場所でセル上を縦に通ります。3本の電源には静的電流を運べる幅と給電点が必要です。

配置の図は加算器の桁ごとに色を分けています。完成後はPDKのDRCで図形ルールを調べ、LVSで抽出した素子と接続を合成ネットリストに照合します。ビューアでは拡大と移動ができ、直下のリンクから配線済みGDSを保存できます。





信号順の初期配置、配置配線後のGDS、HPWLの変化。セル色は加算器の桁、横の配線はチャネル内M1、縦はセル上も通るM2です。

> **ここから分かること。** 図の4トリット加算器は1692 × 1590 µmです。セル端子のHPWLは72.04から48.07 mmへ、33.27%縮みました。DRCは0件、LVSはMatchです。

このGDSから素子接続と配線形状を抽出し、次章の電気的な比較に使います。


In [ ]:
from tritflow.viz.layout import gds_viewer
step("配線済みGDSをビューアで表示")
display(gds_viewer(lay.gds_path, top=net.top))                     # wheel to zoom, drag to pan
gds_download(lay.gds_path)                                                   # the routed GDS itself
s = lay.stats
print(f"{s['instance_count']}セル、{s['row_count']}行、寸法 {s['die_size_um'][0]:.0f} × {s['die_size_um'][1]:.0f} µm"
      f"、利用率 {100*s['utilization']:.0f} %、HPWLの短縮 {s['hpwl_improvement_pct']:.0f} %")


In [ ]:
#@title 6章：配置配線の詳細と任意のフロアプラン実験 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('pnr', '6章：配置配線の詳細と任意のフロアプラン実験', requires=['opening']):
    table([["Metric", "Two-trit adder"]]+[[k,v] for k,v in lay.stats.items()],details="Full physical statistics")
    assert all(check["clean"] for check in lay.data["checks"].values())
    assert lay.stats["routed_nets"] == len(net.nets())
    assert {p["orient"] for p in lay.data["placements"]} <= {"R0","MY"}
    evidence("実行中の加算器 physical checks",lay.data["checks"])
    table([["Initial / final cell-terminal HPWL (µm)",
            [lay.data["history"][i]["hpwl"] for i in (0,-1)]]])
    if EXTENDED:
        show(placement_animation(lay.data))
    if EXTENDED:
        show(routing_progress(lay.data))
    RUN_FLOORPLAN_EXPERIMENT = False  # opt in after changing the controls below
    EXPERIMENT_ROWS, EXPERIMENT_ANNEAL_STEPS, EXPERIMENT_SEED = None, 2400, 7
    if RUN_FLOORPLAN_EXPERIMENT:
        folder = BUILD/"floorplan_experiment"
        def route_experiment():
            routed = net.place_and_route(output_dir=folder, rows=EXPERIMENT_ROWS,
                       anneal_steps=EXPERIMENT_ANNEAL_STEPS, seed=EXPERIMENT_SEED)
            drc, lvs = routed.drc(), routed.lvs()
            assert drc["clean"] and lvs["matched"]
            return {"stats":routed.stats, "drc":drc, "lvs":lvs}
        trial = cache.run("floorplan-experiment", {"net":net.to_json(),"rows":EXPERIMENT_ROWS,
                         "steps":EXPERIMENT_ANNEAL_STEPS,"seed":EXPERIMENT_SEED}, route_experiment,
                         artifacts=[folder/f"{net.top}.gds",folder/f"{net.top}.pnr.json"])
        assert trial["drc"]["clean"] and trial["lvs"]["matched"]
        table([["Measured floorplan experiment",trial["stats"]]])
        show(placement_animation(flow.load_layout(net,folder).data))


## 7. 配線後の電圧と遅延を確かめる

出力が有効なトリットになる時刻を、入力ランプの開始から理想値±0.5 Vに入り、その入力区間の終わりまで留まるまでの時間とします。同じ抽出素子に配線容量を加えた場合と加えない場合を比べ、負荷と桁数を増やします。

### 桁数からの予測

1桁の桁上がり整定時間を$t_{c,1}$、和と桁上がりの遅い方を$t_{e,1}$とすると、リップル経路の予測は

$$
t_{\mathrm{hand}}(N)=t_{e,1}+(N-1)t_{c,1}.
$$

全入力0の静的電力は合成したセル数と各セルのDC電力から足し合わせます。SPICEの電源電流の符号を保って計算します。

$$
P_c(0)=-\sum_{r\in\{DD,MID,SS\}}V_rI_{r,c}(0),\qquad
P_{\mathrm{hand}}(N)=\sum_c n_c(N)P_c(0).
$$





全加算器の27入力状態と負荷掃引。保持500 ns、出力負荷10 fFで、和の最も遅い遷移は配線容量の追加により100.4から108.6 nsへ変わります。負荷掃引は12.81 pFで合格、12.97 pFで不合格です。


In [ ]:
step("同じ素子を配線容量なしでシミュレーション（約30秒）")
pre = lay.extract(wire_c=False).simulate(vectors, load=LOAD_F, expected=adder_reference,
                                         nodeset=guesses, **CONDITIONS)    # same devices, no wire C
for pin, t_post in post["measurements"]["max_settling_s"].items():
    print(f"{pin:7s} 素子のみ {1e9*pre['measurements']['max_settling_s'][pin]:6.1f} ns"
          f" → 配線容量込み {1e9*t_post:6.1f} ns")
step("配線ごとの容量を計算")
wire_report = lay.result.net_parasitics()                                 # routed wire C per logical net
routed_sch = sch.annotate(parasitics=wire_report, title="同じ回路図：定常値と配線容量（fF）")
display(routed_sch)


In [ ]:
#@title 7章：同じ素子で配線容量の有無を比較 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('pre', '7章：同じ素子で配線容量の有無を比較', requires=['opening']):
    assert pre["measurements"]["passed"]
    assert pre["stimulus"]["vectors"] == demo["stimulus"]["vectors"]
    assert pre["stimulus"]["hold_s"] == demo["stimulus"]["hold_s"]
    assert pre["stimulus"]["step_s"] == demo["stimulus"]["step_s"]
    assert routed_sch.scene is sch.scene
    evidence("Schematic wire-C scope (excludes receiving-pin C)",wire_report["assumptions"])
    if EXTENDED:
        for load in [10e-15,1e-12]:
            loaded = cache.run(f"adder2-load-{load:g}", {"vectors":vectors,"load":load}, lambda:
                lay.extract().simulate(vectors, load=load, expected=adder_reference,
                                       hold_s=2e-6, step_s=2e-9, timeout=180))
            assert loaded["measurements"]["passed"], loaded["measurements"]
            print(load, loaded["measurements"]["passed"], loaded["measurements"]["max_settling_s"])




同じパラメータ付きHDLから作る1〜6トリット加算器。上から桁上がり経路の整定時間、測定入力での静的電力の最大値、GDS面積とセル数です。

> **ここから分かること。** 全加算器の27入力状態を500 nsずつ保持した負荷試験では、12.81 pFで合格し、12.97 pFで不合格となります。1〜6桁の桁上がり経路の近似直線は、素子のみで48.98 ns/トリット、配線容量込みで56.88 ns/トリットです。

最大桁数での比較は次のとおりです。遅延は図と同じ和の最上位桁と桁上がりを使い、電力は別の全入力0のDC状態で比べています。

<div style="display:block;width:100%;min-width:0;max-width:100%;overflow-x:auto;contain:inline-size;box-sizing:border-box;">
<table width="100%" style="width:100%;min-width:0;max-width:100%;table-layout:fixed;border-collapse:collapse;">
<tr><th width="33.3333%" style="white-space:normal;overflow-wrap:anywhere;word-break:normal;min-width:0;vertical-align:top;text-align:left;padding:.45em .7em;width:33.3333%;">比較項目</th><th width="33.3333%" style="white-space:normal;overflow-wrap:anywhere;word-break:normal;min-width:0;vertical-align:top;text-align:left;padding:.45em .7em;width:33.3333%;">手計算</th><th width="33.3333%" style="white-space:normal;overflow-wrap:anywhere;word-break:normal;min-width:0;vertical-align:top;text-align:left;padding:.45em .7em;width:33.3333%;">SPICE</th></tr>
<tr><td width="33.3333%" style="white-space:normal;overflow-wrap:anywhere;word-break:normal;min-width:0;vertical-align:top;text-align:left;padding:.45em .7em;width:33.3333%;">素子のみの経路整定時間（ns）</td><td width="33.3333%" style="white-space:normal;overflow-wrap:anywhere;word-break:normal;min-width:0;vertical-align:top;text-align:left;padding:.45em .7em;width:33.3333%;">279</td><td width="33.3333%" style="white-space:normal;overflow-wrap:anywhere;word-break:normal;min-width:0;vertical-align:top;text-align:left;padding:.45em .7em;width:33.3333%;">299.8</td></tr>
<tr><td width="33.3333%" style="white-space:normal;overflow-wrap:anywhere;word-break:normal;min-width:0;vertical-align:top;text-align:left;padding:.45em .7em;width:33.3333%;">全入力0の静的電力（mW）</td><td width="33.3333%" style="white-space:normal;overflow-wrap:anywhere;word-break:normal;min-width:0;vertical-align:top;text-align:left;padding:.45em .7em;width:33.3333%;">400.2</td><td width="33.3333%" style="white-space:normal;overflow-wrap:anywhere;word-break:normal;min-width:0;vertical-align:top;text-align:left;padding:.45em .7em;width:33.3333%;">400.2</td></tr>
</table>
</div>


In [ ]:
#@title 7章：保存済みの桁数・負荷掃引 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('published', '7章：保存済みの桁数・負荷掃引', requires=[]):
    from tritflow.sim.scaling import load_scaling
    from tritflow.sim.load_sweep import load_sweep_results
    from tritflow.viz.scaling import scaling_figure, load_sweep_figure, scaling_fit_figure
    for filename, loader, figure in [("adder_scaling.json",load_scaling,scaling_figure),
                                      ("full_add_load_sweep.json",load_sweep_results,load_sweep_figure)]:
        published = ROOT/"results"/filename
        result = loader(published)
        records = result["rows"] if filename == "adder_scaling.json" else [result]
        compatible_sources = []
        for record in records:
            compatible_sources.extend(verify_publication_sources(record,ROOT,PDK_ROOT))
        if compatible_sources:
            evidence("Recorded-source compatibility: exact optional schematic delegate only",compatible_sources)
        evidence("Published experiment scope: "+filename,result.get("coverage",result.get("scope")))
        evidence("Published limits / conditions",result.get("limits",result.get("conditions")))
        if filename == "adder_scaling.json":
            table([["Width", "Routing realization", "Recorded state", "PASS", "Error"]]+
                  [[r["n"],f"width-sweep/adder{r['n']} · seed {r['seed']}",r["state"],r["passed"],r["error"]] for r in result["rows"]])
        else:
            table([["Measured boundary (F)",result["boundary"]], ["Boundary found",result["boundary_found"]],
                   ["出力s / input-state count",[result["outputs"],len(result["vectors"])]]])
        if EXTENDED:
            show(figure(result))
        if filename == "adder_scaling.json":
            if EXTENDED:
                show(scaling_fit_figure(result))


### STAの予測とSPICEの比較

三値STAは変化前後の電圧の中点を使い、入力と出力の遷移方向、他の入力値、スルーに応じて遅延をつなぎます。最後の出力だけに中点から許容帯までの整定時間を加えます。下のコードは実行中の加算器の入力遷移、配線容量、外部負荷をSPICEと揃えて比べます。


In [ ]:
from tritflow.sta import analyze
states = [dict(zip(inputs, v)) for v in vectors]
wire_caps = {name: e["wire_capacitance_f"] for name, e in wire_report["nets"].items()}
step("静的タイミング解析（SPICEなし）")
timing = analyze(net, vector_pairs=list(zip(states, states[1:])), wire_caps=wire_caps,
                 output_load_f=LOAD_F, input_transition_s=EDGE_S)    # milliseconds, no SPICE
for pin, t_sta in timing.arrival.items():
    print(f"{pin:7s} STA {1e9*t_sta:6.1f} ns   SPICE {1e9*post['measurements']['max_settling_s'][pin]:6.1f} ns")


In [ ]:
#@title 7章：STAの比較表とクリティカルパス { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('sta', '7章：STAの比較表とクリティカルパス', requires=['opening']):
    rows = [[pin, t*1e9, post["measurements"]["max_settling_s"][pin]*1e9,
             100*(t/post["measurements"]["max_settling_s"][pin]-1)] for pin, t in timing.arrival.items()]
    table([["出力", "STA (ns)", "配線後SPICE（ns）", "差（%）"]]+rows)
    assert set(timing.arrival) == set(post["measurements"]["max_settling_s"])
    evidence("STA critical path and assumptions", {"critical_output": timing.critical_output,
             "critical_path": timing.critical_path, "assumptions": timing.assumptions})




同じ入力遷移集合に対するSTAと抽出後SPICEの最大出力遅延。対角線は一致、帯は±20%です。39出力のうち38出力が帯内に入り、平均絶対百分率誤差は6.776%です。

> **ここから分かること。** 保存済みの比較では39出力のうち38出力がSPICEの±20%以内で、平均絶対百分率誤差は6.776%です。例外の`scaling/adder2/cout`は36.43%の過大評価で、イベントモデルにない部分的なパルス振幅と履歴が関係します。


## 8. 自分のHDLでGDSを作る

最初の編集例はmin/maxソーターです。minを`~((~a) | (~b))`で書き、合成した回路図、配線済みGDS、抽出後の波形まで実行します。コードの`CHOICE`を変えると符号検出器、比較器、乗算器も選べます。

等価な書き方を試す間は整数の参照関数を保ち、意図する機能を変えるときは参照関数も更新してください。例えばソーターのminをmaxへ変えると、配置配線に進む前に真理値表の照合が不一致の入力を示します。


In [ ]:
# Your circuit: edit this TritVerilog, then run the cells below.
# This first edit writes min(a, b) through De Morgan, as -max(-a, -b).
source = """
module minmax(input  trit a, b,
              output trit lo, hi);
  assign lo = ~((~a) | (~b));   // min(a, b)
  assign hi = a | b;            // max(a, b)
endmodule
"""


In [ ]:
from examples.curated import DESIGNS
from tritflow.synth import evaluate_netlist
CHOICE = os.environ.get("TRITFLOW_CIRCUIT", "minmax")   # or: sign | compare2 | mul2
step(f"選んだ回路：{CHOICE}")
example = DESIGNS[CHOICE]                                # its independent arithmetic reference
if CHOICE != "minmax":
    source = example.path.read_text()
print(display_label(example.description))


In [ ]:
step("編集したHDLを合成")
custom_circuit = load_edited(example, source, BUILD/"your_circuit"/"source"/example.top)
custom = custom_circuit.synthesize()                                     # your HDL → cells
display(custom.schematic(net_names=True, title=f"{custom.top}：編集したHDLの回路図"))


In [ ]:
#@title 8章：全入力を整数の参照関数と照合 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('custom_check', '8章：全入力を整数の参照関数と照合', requires=[]):
    truth = custom_circuit.truth_table()
    pins = custom_circuit.design.inputs
    planned = list(zip(*(truth[p] for p in pins)))
    oracle = example.oracle(pins)
    mapped_truth = evaluate_netlist(custom,{p:truth[p] for p in pins})
    for i, vector in enumerate(planned):
        expected = oracle(vector)
        source_actual = {p:int(truth[p][i]) for p in expected}
        mapped_actual = {p:int(mapped_truth[p][i]) for p in expected}
        if source_actual != expected or mapped_actual != expected:
            raise AssertionError(f"Independent reference disagreement at {dict(zip(pins,vector))}: "
                                 f"expected {expected}, HDL {source_actual}, cells {mapped_actual}. "
                                 "Check your edit; change the arithmetic reference only if you intend a different function.")
    details_html("編集した回路の全入力真理値表",truth_table_html(custom,max_rows=len(planned)))


In [ ]:
from tritflow.viz.layout import gds_viewer
custom_out = BUILD/"your_circuit"/example.top
step("編集した回路を配置配線")
custom_lay = custom.place_and_route(output_dir=custom_out, seed=0, anneal_steps=600)   # → GDS
step("編集した回路のDRCとLVS")
print("DRC合格：", custom_lay.drc()["clean"], "· LVS一致：", custom_lay.lvs()["matched"])
display(gds_viewer(custom_lay.gds_path, top=custom.top))             # live KLayout in Colab
gds_download(custom_lay.gds_path)                                            # your circuit as GDSII


In [ ]:
#@title 8章：編集したGDSの抽出後SPICEと波形 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('custom', '8章：編集したGDSの抽出後SPICEと波形', requires=['custom_check']):
    custom_result = cache.run(f"your-{CHOICE}", {"source":source,"net":custom.to_json(),"vectors":planned,
                             "expected":[oracle(v) for v in planned],"load_f":10e-15,"hold_s":1e-6,
                             "step_s":2e-9,"rows":None,"anneal_steps":600,"seed":0,"initialize_internal":True},
        lambda: complete_design(custom,custom_out,planned,oracle,hold_s=1e-6,initialize_internal=True),
        artifacts=lambda r: experiment_artifacts(r,custom_out,example.top))
    assert_experiment(custom_result)
    assert {tuple(row["input_trits"]) for row in custom_result["post"]["measurements"]["rows"]} == set(planned)
    table([["整数の参照関数と照合した入力状態数",len(planned)], ["物理結果",custom_result["stats"]],
           ["DRC合格 / LVS一致",[custom_result["drc"]["clean"],custom_result["lvs"]["matched"]]],
           ["Newton法の初期推定値（V）",custom_result["initial_nodeset_v"]]],details="選択した回路の物理検査と初期推定値")

    show(waveform_scope(custom_result["post"],title=f"{CHOICE}：全入力を通る配線後の過渡波形"))
    custom_values = {"custom.name":custom.top,"custom.logic_states":len(planned),
        "custom.cells":custom.report()["total_cells"],"custom.die_um":dimensions(custom_result["stats"]["die_size_um"]),
        "custom.drc_items":custom_result["drc"]["violations"],"custom.lvs_status":"Match",
        "custom.analog_states":len(set(planned)),"custom.analog_status":"PASS"}
    narrative_directory = "narrative_ja" if globals().get("NOTEBOOK_LANGUAGE") == "ja" else "narrative"
    result_marker = "> **ここから分かること。**" if narrative_directory == "narrative_ja" else "> **What this shows.**"
    custom_takeaway = (ROOT/"notebook"/narrative_directory/"08_your_circuit.md").read_text().split(result_marker,1)[1]
    display(Markdown(render_narrative(result_marker+custom_takeaway,{**narrative_values,**custom_values})))


信号に未対応の演算を書くと、合成前に場所と理由を示します。


In [ ]:
from tritflow.hdl import HDLError
broken = """
module broken(input trit a, b, output trit y);
  assign y = a + b;   // '+' on signals is not a gate
endmodule
"""
step("未対応の式を含むHDLを読み込み")
try:
    flow.loads(broken, "broken", filename="your_circuit.tv")
except HDLError as error:
    print(error)
    print("信号の演算はゲートで記述してください。nany(a, b)は飽和加算の符号反転です。")
else:
    raise AssertionError("unsupported signal arithmetic must be rejected")


## 9. GDSからシリコンとの比較へ

元のセルを載せたチップは2027年1月に届く予定で、まだ実測していません。採択された場合は、2027年2月14〜18日のISSCCポスターで、このセルの最初の実測結果とノートブックの予測を並べる計画です。

まずDC入力掃引で伝達特性と全電源の静的電流を測ります。次に入力エッジと負荷を記録して各レベル間の整定時間を測り、同じ条件のSPICEと比べます。電源電圧、温度、パッド・パッケージ・測定器の容量も記録します。


### シャトル向けトップの生成

配線済みコアにフレームと端子を加えるには、`lay.result.to_chip()`と、返されたチップの`signoff()`を使います。下のビューアとダウンロードは、図と同じ加算器をMPWのTEGフレームへ入れたGDSです。

```python
chip = lay.result.to_chip()
checks = chip.signoff()
chip.gds_path
```


In [ ]:
from tritflow.viz.layout import gds_viewer
step("シャトル向けトップのGDSを表示（2.5 mm角、保存済みの実行結果）")
chip = json.loads((ROOT/"notebook/assets/chip/adder4/signoff.json").read_text())   # recorded run, adder4 seed 7
display(gds_viewer(ROOT/"notebook/assets/chip/adder4/tr_1um_adder4.gds"))   # zoom down to devices
gds_download(ROOT/"notebook/assets/chip/adder4/tr_1um_adder4.gds",
             url=f"https://github.com/PenguinEino/tritflow/raw/{SOURCE_COMMIT}/notebook/assets/chip/adder4/tr_1um_adder4.gds")
table([["Check", "Result"]] + [[k, chip[k]] for k in
      ("drawing_drc", "drawing_lvs", "mask_drc", "mpw_precheck", "submission_ready", "reason")])
# Reproduce locally with KLayout and the OpenSUSI MPW template (see docs/status/PNR.md):
#   full = macro.to_chip(); full.signoff()


In [ ]:
#@title 9章：保存済みのトップ検査とパッド図 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('shuttle', '9章：保存済みのトップ検査とパッド図', requires=[]):
    assert chip["physical_checks_passed"] and chip["drawing_drc"]["clean"] and chip["mask_drc"]["clean"]
    assert chip["drawing_lvs"]["matched"] and chip["drawing_lvs"]["strict_ports"]
    assert hashlib.sha256((ROOT/"notebook/assets/chip/adder4/tr_1um_adder4.gds").read_bytes()).hexdigest() == chip["gds_sha256"]
    details_html("パッド図と提出までの残作業",
                 "<pre style='white-space:pre-wrap'>"+(ROOT/"notebook/assets/chip/adder4/README.md").read_text()+"</pre>")


> **ここから分かること。** 保存済みトップは2500 × 2500 µm、23パッドです。描画GDSのDRCは0件、厳密な端子照合を含むLVSはMatch、マスク処理後のDRCは0件、MPW事前チェックはPassです。

実際にシャトルへ出すには、領域の割当やESDの確認などが残ります（付録A）。

ここまでに作った配線済みGDSとライブラリは、別のHDLでも使えます。付録Cにはローカル実行と拡張の手順を載せています。


## 付録A. 適用範囲

**実測.** 元のセルはテープアウト済みで、シリコンの測定はまだありません。

**動作条件.** 特記のない結果は公称モデルです。PVT全域や製造ばらつきに対する保証はありません。

**言語と合成.** 組合せ回路が対象です。セルの機能的完全性は、任意の真理値表を合成する機能を意味しません。

**入力の網羅性.** 論理照合とアナログ試験の範囲は別です。大きい回路のSPICEは代表入力と選択した経路を使い、全入力遷移を網羅していません。

**配線モデル.** 素子と接続はPDKのLVSデッキで抽出します。追加する配線容量は、仮定した係数と面積・周囲長によるネットごとの集中容量です。校正済みPEXではなく、信号配線の抵抗、結合、分布RC、パッケージを含みません。

**手計算とSTA.** 分圧のオン抵抗は近似です。STAは部分パルスの振幅と履歴を表さず、保守的推定も任意の波形の上限を保証しません。

**電源と電力.** 抵抗分圧セルは静的電力を使います。レール抵抗網の計算は理想電源のSPICEとは別です。電流の余裕を見ても、寿命、過渡電流、パッド、ボンド、電流集中までの適合性は確定しません。

**ポリ抵抗.** 任意実験のポリ抵抗は仮定したシート抵抗による感度解析です。校正した分布モデルではありません。

**製造.** コアとフレーム付きトップの検査を区別します。トップでは描画DRC/LVS、マスクDRC、MPW事前チェックを行いますが、マスク処理後のLVSは定義されていません。受動パッドにESD回路はなく、割当、全電源のボンド、VSSの基板・ダイ接着、負荷と電流の確認、プロセス・ESDの承認、最終CIが必要です。


## 付録B. 検証結果と再実行

下のセルは、セルの特性、4つの回路の物理検査と遅延、この実行で計算した結果と保存済みの結果の内訳を表にします。表はすべて折りたたんであります。

<details>
<summary>実行環境と再計算</summary>

Colabではセットアップセルの`TRITFLOW_REF`が指定するコミットを`/content/tritflow`へ取得します。PDKとネイティブツールは`notebook/setup_pins.json`に固定し、Python環境とフォントも確認します。章の途中で失敗したときは、その章の前提となるセルを再実行してください。

`TRITFLOW_FORCE_RUN=1`で重い処理と4回路の保存結果を使う処理を再計算します。これは別途保存しているセル特性、寸法、桁数、負荷の掃引を再生成しません。追加実験は`TRITFLOW_EXTENDED=1`、出力先は`TRITFLOW_NOTEBOOK_BUILD`で指定します。

```sh
python3 -m tritflow.sim.sizing
python3 examples/postlayout_all.py
python3 examples/adder_scaling.py --jobs 1
python3 -m tritflow.sim.publication
python3 -m pytest tests/sim tests/extract -q
```

既存のSPICE出力からSTAの比較だけを再計算するには`python3 -m tritflow.sta.validate`を使います。トップの`chip.signoff(force=True)`は全体の検査を再実行します。各設計のそばにDRC/LVSレポート、SPICEデッキ、ログ、波形を保存します。

</details>

<details>
<summary>出典とライセンス</summary>

設計ソースは[TritFlow](https://github.com/PenguinEino/tritflow)、[著者の三値セル](https://github.com/PenguinEino/opensusi-tr10-balanced-ternary-logic)、[OpenSUSI TR-1um](https://github.com/OpenSUSI/TR-1um)です。KLayoutは[公式配布](https://www.klayout.de/build.html)を使います。TritFlowと著者が公開した回路図はApache-2.0です。PDKのライセンスと帰属表示はNOTICEに残しています。

</details>


In [ ]:
#@title 付録B：セル特性とファンアウトの試験 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('characterization', '付録B：セル特性とファンアウトの試験', requires=[]):
    characters = {cell: json.loads((CELLS_DIR/f"out/{cell}/{cell}.char.json").read_text())
                  for cell in ["INV","NAND","NOR","NANY","BUF"]}
    rows = [["Cell", "Max DC error (mV)", "Max static rail |I| (µA)", "Input C (fF)", "Suggested fanout"]]
    for cell, char in characters.items():
        assert hashlib.sha256((CELLS_DIR/f"out/{cell}/{cell}.ext.spice").read_bytes()).hexdigest() == char["source_sha256"]
        assert all(hashlib.sha256((PDK_ROOT/"libs.tech/spice/models"/name).read_bytes()).hexdigest() == value
                   for name,value in char["model_sha256"].items()), cell
        assert char["passed"] and all(row["passed"] for row in char["dc"])
        assert len({tuple(row["input_trits"]) for row in char["dc"]}) == 3**len(char["inputs"])
        assert all(result["passed"] for result in char["transient"])
        rows.append([cell, 1e3*max(abs(v) for row in char["dc"] for v in row["error_v"].values()),
                     max(abs(i) for row in char["dc"] for i in row["rail_current_a"].values())*1e6,
                     char["input_capacitance"]["max_f"]*1e15, char["fanout"]["suggested_max"]])
    characterization_rows = rows
    evidence("Full-precision DC checkpoints", {cell:char["dc"] for cell,char in characters.items()})
    table([["Cell", "Recorded conditions", "Tested loads (fF)", "Fanout evidence"]]+
          [[cell, char["conditions"], [r["load_f"]*1e15 for r in char["transient"]], char["fanout"]]
           for cell,char in characters.items()], details="Characterization conditions and complete fanout evidence")
    fanout_rows = [["Cell", "Measured receiver counts", "Chain depth", "Worst chain (ns)",
                    "Driver at largest tested fanout (ns)", "Max level error (mV)"]]
    for cell, char in characters.items():
        cascade = char["cascade"]
        assert cascade["source_sha256"] == char["source_sha256"]
        assert cascade["passed"] and cascade["chains_passed"] and all(c["passed"] for c in cascade["cases"])
        largest = max(cascade["fanouts_tested"])
        chain_times = [t for c in cascade["cases"] if c["mode"] == "chain"
                       for t in c["max_settling_s"].values() if t is not None]
        loaded_times = [c["max_settling_s"]["y0"] for c in cascade["cases"]
                        if c["mode"] == "fanout" and c["fanout"] == largest]
        fanout_rows.append([cell, cascade["fanouts_tested"], cascade["chain_depth"],
                           max(chain_times)*1e9, max(loaded_times)*1e9,
                           max(c["max_error_v"] for c in cascade["cases"])*1e3])


In [ ]:
#@title 付録B：4回路の物理検査と遅延の比較 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('four', '付録B：4回路の物理検査と遅延の比較', requires=[]):
    from tritflow.sim.postlayout import representative_vectors, arithmetic_oracle
    FOUR_DESIGN_LOAD_F, FOUR_DESIGN_HOLD_S, FOUR_DESIGN_STEP_S = 10e-15, 500e-9, 2e-9
    four_results = {}
    physical_rows = [["Design", "Routing realization", "寸法（µm）", "Aspect", "Utilization", "DRC / LVS", "Input states visited / possible",
                      "Post max error (mV)", "Rail deviation / zero shift (mV)"]]
    timing_rows = [["Design", "Routing realization", "出力", "Device-only max (ns)", "+ wire-C max (ns)", "Largest same-transition change (ns)"]]
    four_run_labels = {}
    for name, source_name, params in [("adder4","adder",{"N":4}), ("full_add","full_add",{}),
                                      ("mac1","mac1",{}), ("mul1","mul1",{})]:
        mapped = flow.load(ROOT/f"examples/{source_name}.tv", source_name, params).synthesize(name=name)
        pins = [p.name for p in mapped.ports if p.dir == "input"]
        planned, labels = representative_vectors(name, pins)
        oracle = arithmetic_oracle(name, pins)
        directory = BUILD/"four_designs"/name
        options = {"net":mapped.to_json(),"vectors":planned,"labels":labels,
                   "expected":[oracle(v) for v in planned],"load_f":FOUR_DESIGN_LOAD_F,
                   "hold_s":FOUR_DESIGN_HOLD_S,"step_s":FOUR_DESIGN_STEP_S,
                   "rows":None,"anneal_steps":600,"seed":0,"initialize_internal":True}
        lookup_started = time.monotonic()
        result = None if cache.force else recorded_campaign.get(name,options)
        if result is not None:
            cache.events.append((f"four-{name}","recorded"))
            cache.timings.append({"stage":f"four-{name}","state":"recorded",
                                  "elapsed_s":time.monotonic()-lookup_started})
            step(f"{name}：保存済みのレイアウトとSPICE結果を使用（検査は下の表）")
        else:
            step(f"{name}：配置配線、DRC/LVS、SPICE（1〜3分）")
            result = cache.run(f"four-{name}",options,
                lambda: complete_design(mapped,directory,planned,oracle,load_f=FOUR_DESIGN_LOAD_F,
                           hold_s=FOUR_DESIGN_HOLD_S,step_s=FOUR_DESIGN_STEP_S,initialize_internal=True,dataset='four-design'),
                artifacts=lambda r: experiment_artifacts(r,directory,name))
        assert_experiment(result)
        state = 'recorded' if cache.events[-1][1] == 'recorded' else ('fresh' if cache.events[-1][1] == 'run' else 'local reuse')
        if state == 'recorded':
            assert result['routing'] == archive_routing(ROOT/'notebook/assets/four_designs.zip')[name]
        run_label = routing_label(result['routing'],state=state,digest=True)
        four_run_labels[name] = run_label
        assert result["post"]["stimulus"]["vectors"] == list(map(tuple,planned))
        stats, ir = result["stats"], result["ir"]
        rail = max(abs(v) for c in ir["cells"].values() for v in c["rail_deviation_v"].values())
        zero = max(abs(c["zero_level_shift_v"]) for c in ir["cells"].values() if c["zero_level_shift_v"] is not None)
        physical_rows.append([name,run_label,stats["die_size_um"],max(stats["die_size_um"])/min(stats["die_size_um"]),
                              stats["utilization"],[result["drc"]["clean"],result["lvs"]["matched"]],
                              f"{len(set(planned))} / {3**len(pins)}",result["post"]["measurements"]["max_error_v"]*1e3,
                              [rail*1e3,zero*1e3]])
        before, after = [result[mode]["measurements"] for mode in ("pre","post")]
        for pin in after["max_settling_s"]:
            changes = [b["outputs"][pin]["settling_s"]-a["outputs"][pin]["settling_s"]
                       for a,b in zip(before["rows"][1:],after["rows"][1:],strict=True)]
            timing_rows.append([name,run_label,pin,before["max_settling_s"][pin]*1e9,after["max_settling_s"][pin]*1e9,
                                max(changes)*1e9])
        four_results[name] = result


    if recorded_campaign.manifest:
        evidence("Recorded four-design producer tools (not local requalification)",
                 recorded_campaign.manifest["tools"])
    table([["Shared load (fF)",FOUR_DESIGN_LOAD_F*1e15], ["Hold / max step (ns)",
           [FOUR_DESIGN_HOLD_S*1e9,FOUR_DESIGN_STEP_S*1e9]],
           ["Input plan labels", {name:representative_vectors(name,
              r["post"]["stimulus"]["inputs"])[1] for name,r in four_results.items()}]],details="Four-design stimulus conditions and labels")


In [ ]:
#@title 付録B：検証表、入力データ、実行時間 { display-mode: "form" }
if "_tf_cells" not in globals():
    raise RuntimeError("Run setup and imports successfully before this chapter; rerun them after any requested runtime restart.")
with _tf_cells.running('final', '付録B：検証表、入力データ、実行時間', requires=['opening_inputs', 'opening', 'gates', 'completeness', 'sizing', 'cells', 'arithmetic', 'synthesis', 'pnr', 'pre', 'published', 'sta', 'custom_check', 'custom', 'shuttle', 'characterization', 'four']):
    table(characterization_rows,details="Cell DC / capacitance / fanout summary")
    table(fanout_rows,details="Transistor-level cascade and fanout tests")
    table(cell_dimension_rows,details="All standard-cell dimensions and pin-access tracks")
    table(cell_qualification_rows,details="Cell drawing-DRC and LVS qualification")
    table(physical_rows,details="Four-design physical and arithmetic checks (retained or fresh as below)")
    table(timing_rows,details="Four-design identical-device timing comparison")
    evidence("Cell ordered-pair/orientation abutments",report["abutment"])
    for name,grid in sizing_grids.items():
        evidence(name+" sizing status including unavailable points",[
            {"index":i,"dimensions":r["dimensions"],"passed":r["passed"],"error":r["error"]}
            for i,r in enumerate(grid["rows"]) if not r["passed"]])
    evidence("実行中の加算器 acceptance, sequence and initialized Newton guesses",{
        "logic":net.report()["equivalence"],"labels":vector_labels,"vectors":vectors,
        "measurements":demo["measurements"],"nodeset":demo["initial_nodeset_v"],
        "conditions":{"load_f":LOAD_F,"hold_s":HOLD_S,"edge_s":EDGE_S,"step_s":STEP_S,"temperature_c":TEMP_C}})
    evidence("Native paper-figure inputs and committed image/data hashes",paper)
    table([["Routing realization", "PNR SHA-256", "GDS SHA-256"]]+
          [[routing_label(r),r['pnr_sha256'],r['gds_sha256']]
           for r in (paper['paper_pnr']['routing'],paper['paper_post']['routing'])]+
          [[four_run_labels[name],r['routing']['pnr_sha256'],r['routing']['gds_sha256']]
           for name,r in four_results.items()],details="Native identities for each paper and four-design routing realization")
    assert (runtime_fingerprint(ROOT,PDK_ROOT,names=native_sources) == cache.fingerprint and
            source_manifest(ROOT,PDK_ROOT,names=native_sources) == native_sources), "consumed sources changed during Run all; rerun after publication settles"
    load_paper(ROOT)  # measurements remain strict; illustration changes only warn
    table([["Stage", "Fresh / local cache / recorded", "Wall time this run (s)"]]+[
        [r["stage"],r["state"],round(r["elapsed_s"],3)] for r in sorted(cache.timings,key=lambda r:r["elapsed_s"],reverse=True)])
    table([["Design", "Routing realization", "Measured producer phase", "Wall time (s)", "Process + child CPU time (s)"]]+[
        [name,four_run_labels[name],stage,round(seconds,3),round(result["cpu_times_s"][stage],3)] for name,result in four_results.items()
        for stage,seconds in result["timings_s"].items()],details="Physical and pre/post-SPICE runtime breakdown (producer evidence)")
    print("各処理の実行方法：")
    for name,state in cache.events:
        print(f"  {name}: {state}")
    print(f"セットアップを含む実行時間：{time.monotonic()-started:.1f} s")
    print("生成ファイル：", BUILD)


## 付録C. 再利用と拡張

### ローカルで実行する

空の作業ディレクトリから、ソースと同梱データを含むチェックアウトを作ります。Pythonパッケージだけではセルや例題のファイルは揃いません。

```sh
python3 -m pip install --src ./src -e "git+https://github.com/PenguinEino/tritflow.git@colab-v0#egg=tritflow"
cd src/tritflow
bash notebook/colab_setup.sh --source "$PWD"
```

セットアップは対応するUbuntu amd64環境向けです。既存ツールを使う場合は`TR1UM_PDK`と、必要なら`KLAYOUT_BINARY`を設定します。対応環境は`notebook/README.md`を参照してください。

```python
from tritflow import flow
net = flow.load("examples/adder2.tv", "adder2").synthesize()
layout = net.place_and_route(output_dir="build/adder2")
assert layout.drc()["clean"] and layout.lvs()["matched"]
vectors = [tuple(0 for p in net.ports if p.dir == "input")]
post = layout.extract().simulate(vectors, load=100e-15)
```

この例は全入力0の動作確認です。`load`はF単位、入力の順序は`net.ports`です。算術の検証では独立した参照関数を`expected=`へ渡し、調べる経路を通る入力遷移を指定します。結果は`post["measurements"]`で確認します。`extract(wire_c=False)`は追加の配線容量を外します。

配置前の回路図は`net.schematic(expand="*")`、文字列のHDLは`flow.loads(source, top=..., params=...)`で扱えます。CLIからも合成と配置配線を実行できます。

```sh
python3 -m tritflow.synth examples/adder2.tv --top adder2 -o build/adder2
python3 -m tritflow.pnr build/adder2/adder2.json -o build/adder2 --seed 7
```

### セルを追加する

回路・端子・真理値を`tritflow/cells/circuits.py`、形状を`compact.py`、PDK素子の呼出しを`primitives.py`で定義します。共通高さ、格子、電源順、ピンの接続位置を保ち、内部の配線障害物を抽象情報へ含めます。

合成には`tritflow/synth/simulate.py`の評価と`mapping.py`の割当規則、SPICEには`tritflow/sim/spice.py`のピン順と`characterize.py`の論理値を追加します。HDLプリミティブならparserとelaborator、STAを使うなら`tritflow/sta/logic.py`と遷移特性、回路図には`tritflow/schematic/render.py`も更新します。

配布セルを置き換える前に試作用の出力先で生成し、独立した真理値照合、DRC/LVS、DC、負荷付き過渡解析、突き当て配置、試験回路の配線を確認してください。

```sh
python3 -m tritflow.cells build --out build/cell_trial
python3 -m tritflow.cells verify --out build/cell_trial
python3 -m tritflow.sim.characterize INV --spice build/cell_trial/INV/INV.ext.spice --destination build/INV.char.json
```

### 演算子を追加する

既存演算のHDLモジュールで書けるかを先に検討します。構文を増やす場合は`tritflow/hdl/parser.py:GRAMMAR`、`elaborate.py:Elaborator.expression`、`ir.py:Design.evaluate`を更新し、`tritflow/synth/optimize.py`に三値で成立する書換え、`mapping.py:Mapper.map`に割当規則を追加します。

小さい演算子は全入力で独立した定義と照合し、不正な記述の診断も試します。影響するテストに加え、新しい素子を使う場合は抽出デッキも見直してください。

```sh
python3 -m pytest tests/hdl tests/synth -q
python3 -m pytest tests/cells tests/pnr tests/sim tests/extract -q
python3 -m pytest tests/sta tests/schematic -q
```
